# Project 66 — Local Groundedness Checker
## Sentence-Level Grounding Analysis for RAG Outputs

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Test Cases with Context & Answers

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import pandas as pd, json

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

test_pairs = [
    {
        "context": "Python was created by Guido van Rossum and first released in 1991. "
                   "It emphasizes code readability with significant whitespace. "
                   "Python supports multiple paradigms: procedural, OOP, and functional.",
        "answer": "Python was created by Guido van Rossum in 1991. It's the most popular "
                  "programming language in the world and emphasizes readability. "
                  "It supports OOP, procedural, and functional programming."
    },
    {
        "context": "Docker containers share the host OS kernel and are lighter than VMs. "
                   "Images are built from Dockerfiles. Containers are ephemeral by default.",
        "answer": "Docker uses containers that share the OS kernel. They're lighter than VMs. "
                  "Docker was created in 2013 by Solomon Hykes at dotCloud. "
                  "Kubernetes manages Docker containers at scale."
    },
    {
        "context": "PostgreSQL is an open-source relational database. It supports JSON, "
                   "full-text search, and ACID transactions. Default port is 5432.",
        "answer": "PostgreSQL is an open-source database supporting JSON and ACID. "
                  "It runs on port 5432 by default. It's faster than MySQL for complex queries."
    },
]
print(f"Test pairs: {len(test_pairs)}")

## Step 2 — Sentence-Level Grounding Check

In [ ]:
class SentenceGround(BaseModel):
    sentence: str
    grounded: bool
    evidence: str = Field(description="Supporting quote from context, or 'none'")
    category: str = Field(description="fully_supported, partially_supported, unsupported, fabricated")

class GroundednessReport(BaseModel):
    total_sentences: int
    grounded_count: int
    ungrounded_count: int
    score: float = Field(ge=0, le=1)
    sentences: list[SentenceGround]

checker = llm.with_structured_output(GroundednessReport)

reports = []
for i, pair in enumerate(test_pairs):
    report = checker.invoke(
        f"Check each sentence in the ANSWER for grounding in the CONTEXT.\n\n"
        f"CONTEXT:\n{pair['context']}\n\nANSWER:\n{pair['answer']}"
    )
    reports.append(report)
    print(f"\nPair {i+1}: score={report.score:.0%} ({report.grounded_count}/{report.total_sentences} grounded)")
    for s in report.sentences:
        icon = "✓" if s.grounded else "✗"
        print(f"  {icon} [{s.category}] {s.sentence[:60]}")
        if not s.grounded:
            print(f"    Evidence: {s.evidence}")

## Step 3 — Aggregated Metrics

In [ ]:
rows = []
for i, report in enumerate(reports):
    for s in report.sentences:
        rows.append({
            "pair": i + 1,
            "sentence": s.sentence[:50],
            "grounded": s.grounded,
            "category": s.category,
        })

df = pd.DataFrame(rows)
print("GROUNDEDNESS SUMMARY")
print("=" * 50)
print(f"Total sentences analyzed: {len(df)}")
print(f"Grounded:   {df['grounded'].sum()} ({df['grounded'].mean():.0%})")
print(f"Ungrounded: {(~df['grounded']).sum()}")
print(f"\nBy category:")
print(df["category"].value_counts().to_string())
print(f"\nPer-pair scores:")
for i, report in enumerate(reports):
    print(f"  Pair {i+1}: {report.score:.0%}")

## Step 4 — Improvement Suggestions

In [ ]:
improve_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given ungrounded sentences, suggest how to fix the RAG answer "
     "to only contain information from the context. Be specific."),
    ("human", "Context: {context}\n\nUngrounded sentences: {ungrounded}")
])
improve_chain = improve_prompt | llm | StrOutputParser()

for i, (pair, report) in enumerate(zip(test_pairs, reports)):
    ungrounded = [s.sentence for s in report.sentences if not s.grounded]
    if ungrounded:
        fix = improve_chain.invoke({
            "context": pair["context"],
            "ungrounded": "\n".join(ungrounded),
        })
        print(f"\nPair {i+1} improvements:")
        print(fix[:300])

## What You Learned
- **Sentence-level grounding** analysis
- **Evidence mapping** from context to claims
- **Category classification**: supported, partial, unsupported, fabricated
- **Auto-improvement suggestions** for RAG outputs